## Building a simple LLM app 

References:
* https://github.com/openai/openai-python
* https://platform.openai.com/docs/api-reference/responses/create


In [4]:
from openai import OpenAI
import os

In [5]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

### Generate text from a simple prompt

In [8]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input = [
            {
                "role": "user",
                "content": "How do I check if a Python object is an instance of a class?"
            }
            ]
)

print(response.output_text)

To check if a Python object is an instance of a certain class, you can use the built-in function `isinstance()`.

### Syntax:
```python
isinstance(object, classinfo)
```

- `object`: The object you want to check.
- `classinfo`: A class, type, or a tuple of classes and types.

### Example:
```python
class MyClass:
    pass

obj = MyClass()

print(isinstance(obj, MyClass))  # Output: True

print(isinstance(obj, int))      # Output: False
```

### Checking against multiple classes:
```python
class A:
    pass

class B:
    pass

obj = A()

print(isinstance(obj, (A, B)))  # Output: True
```

`isinstance()` returns `True` if the object is an instance of the class or any subclass thereof.


### Adding a system prompt prior to user input

In [9]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input = [
            {
                "role": "system",
                "content": "You are a coding assistant that talks like a pirate.",
            },
            {
                "role": "user",
                "content": "How do I check if a Python object is an instance of a class?"
            }
            ]
)

print(response.output_text)


Arrr, matey! To check if a Python object be an instance of a class, ye use the `isinstance()` function. It be easy as finding buried treasure!

Here’s the proper way to do it:

```python
if isinstance(obj, ClassName):
    print("Aye, this object be an instance of ClassName!")
else:
    print("Nay, it be not!")
```

Replace `obj` with yer object’s name and `ClassName` with the class ye be checking against.

Fair winds on yer coding voyage! 🏴‍☠️


### Processing image inputs (Passing a URL)

In [11]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ]
)
print(response.output_text)


This image shows a peaceful rural scene featuring a wooden boardwalk that stretches straight through a lush green field of tall grass and various bushes. Beyond the field, there are clusters of trees in the distance. Above, the sky is mostly blue with some soft, white clouds scattered across it. The lighting suggests it may be late afternoon or early evening, providing a warm and serene atmosphere. This kind of path is often found in nature reserves or parks, guiding visitors through natural landscapes without disturbing the environment.


### Processing image inputs (passing a base64 encoded image)

In [17]:
import base64
# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
# Path to your image
image_path = "sample_image.jpg"

# Getting the Base64 string
encoded_string = encode_image(image_path)
image_url = f"data:image/png;base64,{encoded_string}"
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": image_url,
                    },
                ],
            }
    ]
)
print(response.output_text)

This image shows a wooden boardwalk or pathway cutting through a green meadow or field. The grass on either side of the path is lush and tall, and there are some bushes and trees in the distance. The sky is expansive and mostly clear with some wispy clouds, indicating a bright and sunny day. The overall scene looks peaceful and likely part of a natural or park area. If you want, I can help identify suitable activities for this place or provide a description for various uses.


### Process text and image inputs

In [12]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "what are the objects in this image",
                    },
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ]
)
print(response.output_text)


The image contains the following objects:
- A wooden boardwalk or pathway extending into the distance.
- Dense green grass on both sides of the boardwalk.
- Various trees and bushes in the background.
- A partly cloudy blue sky with scattered clouds.


### Structured output
Specifying output format with json schema

In [29]:
response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "what are the objects in this image",
                    },
                    {
                        "type": "input_image",
                        "image_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg",
                    },
                ],
            }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "object_description",
            "schema": {
                "type": "object",
                "properties": {
                    "objects": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "name": {"type": "string"},
                                "description": {"type": "string"},
                                "color": {"type": "string"},
                            },
                            "required": ["name", "description", "color"],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["objects"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
)
print(response.output_text)


{"objects":[{"name":"boardwalk","description":"a wooden boardwalk path","color":"light brown"},{"name":"grass","description":"tall green grass surrounding the boardwalk","color":"green"},{"name":"trees","description":"trees in the background","color":"green"},{"name":"sky","description":"sky with clouds","color":"blue with white clouds"}]}


In [30]:
import json

def pretty_print_json(json_string):
    try:
        data = json.loads(json_string)
        print(json.dumps(data, indent=2))
    except json.JSONDecodeError as e:
        print("Invalid JSON:", e)
pretty_print_json(response.output_text)

{
  "objects": [
    {
      "name": "boardwalk",
      "description": "a wooden boardwalk path",
      "color": "light brown"
    },
    {
      "name": "grass",
      "description": "tall green grass surrounding the boardwalk",
      "color": "green"
    },
    {
      "name": "trees",
      "description": "trees in the background",
      "color": "green"
    },
    {
      "name": "sky",
      "description": "sky with clouds",
      "color": "blue with white clouds"
    }
  ]
}


### Building Meal Lens

In [25]:
def analyze_food_image(base64_image: str):
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": "You are a helpful culinary assistant that extracts structured recipe data from food images.  Make the output in a warm, wholesome tone like a southern grandma.",
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Extract the dish name, description of the dish, recipe, ingredients, nutrition facts, relevant tags, and suggested food pairings from this image. Keep the description short and sweet. Return the recipe in steps in markdown format (separate each step by a line break). Make the output in a warm, wholesome tone like a southern grandma.",
                    },
                    {
                        "type": "input_image",
                        "image_url": f"{base64_image}",
                    },
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "food_analysis",
                "schema": {
                    "type": "object",
                    "properties": {
                        "dish_name": {"type": "string"},
                        "description": {"type": "string"},
                        "tags": {"type": "array", "items": {"type": "string"}},
                        "recipe": {"type": "string"},
                        "ingredients": {"type": "array", "items": {"type": "string"}},
                        "nutrition_facts": {
                            "type": "object",
                            "properties": {
                                "serving_size": {"type": "string"},
                                "calories": {"type": "integer"},
                                "protein": {"type": "integer"},
                                "carbohydrates": {"type": "integer"},
                                "fat": {"type": "integer"},
                            },
                            "required": [
                                "serving_size",
                                "calories",
                                "protein",
                                "carbohydrates",
                                "fat",
                            ],
                            "additionalProperties": False,
                        },
                        "food_pairings": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                    },
                    "required": [
                        "dish_name",
                        "description",
                        "tags",
                        "recipe",
                        "ingredients",
                        "nutrition_facts",
                        "food_pairings",
                    ],  # Added to required
                    "additionalProperties": False,
                },
                "strict": True,
            }
        },
    )
    return response.output_text

In [23]:
image_url = f"data:image/png;base64,{encode_image("recipe_scallop.jpg")}"

In [32]:
response = analyze_food_image(image_url)

In [33]:
response

'{"dish_name":"Seared Scallops with Pea Puree","description":"Tender, golden-seared scallops served atop a creamy, vibrant pea puree, garnished with fresh herbs and crispy pancetta bits.","tags":["seafood","appetizer","gluten-free","light","elegant","quick"],"recipe":"1. Start by prepping your fresh peas and blanching them in boiling water until tender, then drain and cool.\\n2. Blend the peas with a bit of cream, salt, and pepper until smooth to create your pea puree.\\n3. Pat the scallops dry with a paper towel and season lightly with salt and pepper.\\n4. Heat a pan with a drizzle of oil over medium-high heat until hot.\\n5. Carefully place the scallops in the pan, cooking for about 2-3 minutes on each side until they turn golden brown and are cooked through.\\n6. In the same pan, quickly crisp up some diced pancetta until nice and crunchy.\\n7. Spoon a generous portion of pea puree onto your plate, gently place the scallops on top, and sprinkle with the crispy pancetta and fresh he

In [34]:
pretty_print_json(response)

{
  "dish_name": "Seared Scallops with Pea Puree",
  "description": "Tender, golden-seared scallops served atop a creamy, vibrant pea puree, garnished with fresh herbs and crispy pancetta bits.",
  "tags": [
    "seafood",
    "appetizer",
    "gluten-free",
    "light",
    "elegant",
    "quick"
  ],
  "recipe": "1. Start by prepping your fresh peas and blanching them in boiling water until tender, then drain and cool.\n2. Blend the peas with a bit of cream, salt, and pepper until smooth to create your pea puree.\n3. Pat the scallops dry with a paper towel and season lightly with salt and pepper.\n4. Heat a pan with a drizzle of oil over medium-high heat until hot.\n5. Carefully place the scallops in the pan, cooking for about 2-3 minutes on each side until they turn golden brown and are cooked through.\n6. In the same pan, quickly crisp up some diced pancetta until nice and crunchy.\n7. Spoon a generous portion of pea puree onto your plate, gently place the scallops on top, and spri

### Building Deal to Meal

In [ ]:
# Scrape latest flyer url for each banner from a public website
# For the purpose of demoing building an LLM app, this step is replaced by hard coding flyer urls here.

banner_flyer_dict= {
    "no_frills": ["https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-1.jpg",
                    "https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-2.jpg",
                    "https://flyers.smartcanucks.ca/uploads/pages/272912/no-frills-on-flyer-july-24-to-301-3.jpg"],
    "loblaws": ["https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-1.jpg",
                "https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-2.jpg",
                "https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-3.jpg"],
    "t_t": ["https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-1.jpg",
            "https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-2.jpg",
            "https://flyers.smartcanucks.ca/uploads/pages/272705/tt-supermarket-gta-flyer-july-18-to-24-3.jpg"]
            }
def generate_flyer_dinner(banner):
    urls = banner_flyer_dict[banner]
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Extract all products listed in these flyers. Then, generate one dinner recipe for two people that uses as many of those flyer products as possible. When referencing ingredients from the flyer, match their names exactly as shown. Finally, estimate the total cost of the dinner based on the flyer prices.",
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[0]
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[1]
                    },
                    {
                        "type": "input_image",
                        "image_url": urls[2]
                    },              
                ],
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "flyer_dinner",
                "schema": {
                    "type": "object",
                    "properties": {
                        "dish_name": {"type": "string"},
                        "description": {"type": "string"},
                        "tags": {"type": "array", "items": {"type": "string"}},
                        "recipe": {"type": "string"},
                        "ingredients": {"type": "array", "items": {"type": "string"}},
                        "cost": {"type": "integer"},
                        "nutrition_facts": {
                            "type": "object",
                            "properties": {
                                "serving_size": {"type": "string"},
                                "calories": {"type": "integer"},
                                "protein": {"type": "integer"},
                                "carbohydrates": {"type": "integer"},
                                "fat": {"type": "integer"},
                            },
                            "required": [
                                "serving_size",
                                "calories",
                                "protein",
                                "carbohydrates",
                                "fat",
                            ],
                            "additionalProperties": False,
                        },
                    },
                    "required": [
                        "dish_name",
                        "description",
                        "tags",
                        "recipe",
                        "ingredients",
                        "nutrition_facts",
                        "cost",
                    ],  # Added to required
                    "additionalProperties": False,
                },
                "strict": True,
            }
        },
    )
    return {"llm_response": response.output_text,
            "urls":
            {"url1": urls[0],
            "url2": urls[1],
            "url3": urls[2]}}

In [36]:
response = generate_flyer_dinner("loblaws")
response

{'llm_response': '{"dish_name":"Cheesy Perogies with Broccoli and Grilled Chicken","description":"A delicious dinner for two featuring Cheemo Perogies, steamed PC Vegetables Broccoli Florets, and grilled Chicken Thighs, served with a side of PC Ketchup for dipping.","tags":["dinner","easy","comfort food","Canadian products"],"recipe":"1. Cook Cheemo Perogies as per package instructions.\\n2. Steam PC Vegetables Broccoli Florets until tender.\\n3. Season Chicken Thighs with salt and pepper, then grill or pan-fry until cooked through.\\n4. Serve the chicken thighs alongside the cooked perogies and steamed broccoli.\\n5. Add PC Ketchup as a dipping sauce on the side.","ingredients":["Cheemo Perogies","PC Vegetables Broccoli Florets","Chicken Thighs","PC Ketchup"],"cost":16,"nutrition_facts":{"serving_size":"1 meal for 2 people","calories":650,"protein":45,"carbohydrates":55,"fat":25}}',
 'urls': {'url1': 'https://flyers.smartcanucks.ca/uploads/pages/272915/loblaws-on-flyer-july-24-to-301-

In [37]:
pretty_print_json(response["llm_response"])

{
  "dish_name": "Cheesy Perogies with Broccoli and Grilled Chicken",
  "description": "A delicious dinner for two featuring Cheemo Perogies, steamed PC Vegetables Broccoli Florets, and grilled Chicken Thighs, served with a side of PC Ketchup for dipping.",
  "tags": [
    "dinner",
    "easy",
    "comfort food",
    "Canadian products"
  ],
  "recipe": "1. Cook Cheemo Perogies as per package instructions.\n2. Steam PC Vegetables Broccoli Florets until tender.\n3. Season Chicken Thighs with salt and pepper, then grill or pan-fry until cooked through.\n4. Serve the chicken thighs alongside the cooked perogies and steamed broccoli.\n5. Add PC Ketchup as a dipping sauce on the side.",
  "ingredients": [
    "Cheemo Perogies",
    "PC Vegetables Broccoli Florets",
    "Chicken Thighs",
    "PC Ketchup"
  ],
  "cost": 16,
  "nutrition_facts": {
    "serving_size": "1 meal for 2 people",
    "calories": 650,
    "protein": 45,
    "carbohydrates": 55,
    "fat": 25
  }
}
